In [ ]:
pip install langgraph

In [ ]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

In [ ]:
# Customer Support Chatbot using OpenRouter API (instead of Groq)

from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

#  CONFIG (OpenRouter)
API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

#  Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

#  Define Nodes

def get_answer(state: BotState):
    q = state["question"]

    if not api_key:
        raise ValueError("OpenRouter API key not found in Colab secrets.")

    #  OpenRouter API Call
    response = requests.post(
        API_URL,
        headers=HEADERS,
        json={
            "model": "openai/gpt-3.5-turbo",   # you can change model
            "messages": [
                {"role": "user", "content": q}
            ],
            "temperature": 0.7
        }
    )

    # Error handling
    if response.status_code != 200:
        try:
            error_details = response.json()
        except:
            error_details = response.text
        raise Exception(f"API Error: {error_details}")

    data = response.json()

    # Extract answer
    ans = data["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {
        "answer": f"Bot says: {state['answer']}"
    }

#  Build Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

#  Flow
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# Run Example
if __name__ == "__main__":
    state = {
        "question": "What are your store hours?",
        "answer": "",
        "history": []
    }

    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])
    print("History:", result["history"])

Bot says: I am an AI assistant and do not have store hours. Please contact the specific store you are inquiring about for their hours of operation.
History: ['What are your store hours?']


In [ ]:
# Customer Support Chatbot using OpenRouter API (instead of Groq)

from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# CONFIG (OpenRouter)
API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

#  Define Nodes

def get_answer(state: BotState):
    q = state["question"]

    if not api_key:
        raise ValueError("OpenRouter API key not found in Colab secrets.")

    # 🔥 OpenRouter API Call
    response = requests.post(
        API_URL,
        headers=HEADERS,
        json={
            "model": "openai/gpt-3.5-turbo",   # you can change model
            "messages": [
                {"role": "user", "content": q}
            ],
            "temperature": 0.7
        }
    )

    # Error handling
    if response.status_code != 200:
        try:
            error_details = response.json()
        except:
            error_details = response.text
        raise Exception(f"API Error: {error_details}")

    data = response.json()

    # Extract answer
    ans = data["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {
        "answer": f"Bot says: {state['answer']}"
    }

#  Build Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

#  Flow
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

#  Run Example
if __name__ == "__main__":
    state = {
        "question": "What are your store hours?",
        "answer": "",
        "history": []
    }

    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])
    print("History:", result["history"])

Bot says: I am an AI assistant and do not have physical store hours. If you have any questions or need assistance, feel free to ask at any time.
History: ['What are your store hours?']


In [ ]:
# Scenario: Customer Support Call Center
# A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

# 1. State Definition (SupportState)
# The chatbot keeps track of:
# - query → What the customer asked.
# - category → Which department it belongs to (billing, tech, general).
# - response → What the bot replies with.
# Think of this as the customer’s “ticket form.”

# 2. Router Node (route_query)
# When a customer types a question, the router scans it:
# - If the query mentions “bill” or “payment”, it routes to billing_agent.
# - If it mentions “error” or “bug”, it routes to tech_agent.
# - Otherwise, it defaults to general_agent.
# This is like a receptionist deciding which desk you should go to.

# 3. Agent Nodes
# - billing_agent → Replies with “Billing dept: [query]”.
# - tech_agent → Replies with “Tech support: [query]”.
# - general_agent → Replies with “General help: [query]”.
# Each agent specializes in its own type of problem.

# 4. Graph Flow
# - Entry point: router.
# - Router decides the path based on the query.
# - The query flows into the correct agent node.
# - The agent generates a response and ends the conversation.


from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",
    route_query,
    {
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)

In [ ]:
# Scenario: AI Symptom Tracker (Question-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:
# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"
# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.
# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.
# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.
# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.
# - End Node
# - Delivers the final recommendation to the patient.
from langgraph.graph import StateGraph, END
from typing import TypedDict

#  State
class SupportState(TypedDict):
    query: str
    category: str
    response: str

#  Router function (decides next node)
def route_query(state: SupportState) -> str:
    q = state["query"].lower()

    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

# Dummy router node (required for entry point)
def router_node(state: SupportState):
    return {}

#  Agent Nodes
def billing_agent(state: SupportState):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state: SupportState):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state: SupportState):
    return {"response": "General help: " + state["query"]}

#  Build Graph
g = StateGraph(SupportState)

g.add_node("router", router_node)  # IMPORTANT
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

#  Entry Point
g.set_entry_point("router")

#  Conditional Routing
g.add_conditional_edges(
    "router",
    route_query,
    {
        "billing_agent": "billing_agent",
        "tech_agent": "tech_agent",
        "general_agent": "general_agent",
    }
)

#  End Flow
g.add_edge("billing_agent", END)
g.add_edge("tech_agent", END)
g.add_edge("general_agent", END)

#  Run Example
if __name__ == "__main__":
    app = g.compile()

    state = {
        "query": "I have a payment issue",
        "category": "",
        "response": ""
    }

    result = app.invoke(state)

    print(result["response"])

Billing dept: I have a payment issue


In [ ]:
# Scenario: AI Symptom Tracker (Question-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:
# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"
# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.
# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.
# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.
# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.
# - End Node
# - Delivers the final recommendation to the patient.

from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# API Config (OpenRouter)
API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get("OPENAI_API_KEY")

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

#  State
class HealthState(TypedDict):
    symptom: str
    observations: list
    analysis: str
    recommendation: str
    steps_done: int

# Observation Node (API)
def observation_node(state: HealthState):
    symptom = state["symptom"]

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json={
            "model": "openai/gpt-3.5-turbo",
            "messages": [
                {"role": "user",
                 "content": f"Give one possible observation about symptom: {symptom}"}
            ],
            "temperature": 0.7
        }
    )

    data = response.json()
    obs = data["choices"][0]["message"]["content"]

    return {
        "observations": state["observations"] + [obs],
        "steps_done": state["steps_done"] + 1
    }

# Analysis Node
def analysis_node(state: HealthState):
    combined = " | ".join(state["observations"])
    return {
        "analysis": f"Summary of observations: {combined}"
    }

# Condition Function
def check_condition(state: HealthState):
    if len(state["observations"]) < 3:
        return "observe_again"
    else:
        return "recommend"

# Recommendation Node
def recommendation_node(state: HealthState):
    return {
        "recommendation": f"For {state['symptom']}, monitor symptoms and consult a doctor if it persists."
    }

#  Build Graph
g = StateGraph(HealthState)

g.add_node("observe", observation_node)
g.add_node("analyze", analysis_node)
g.add_node("recommend", recommendation_node)

# Entry
g.set_entry_point("observe")

# Flow
g.add_edge("observe", "analyze")

g.add_conditional_edges(
    "analyze",
    check_condition,
    {
        "observe_again": "observe",
        "recommend": "recommend"
    }
)

g.add_edge("recommend", END)

#  Run
if __name__ == "__main__":
    app = g.compile()

    state = {
        "symptom": "persistent cough",
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0
    }
    result = app.invoke(state)

    print("Observations:", result["observations"])
    print("Analysis:", result["analysis"])
    print("Recommendation:", result["recommendation"])

Observations: ['One possible observation about a persistent cough is that it could be a sign of a respiratory infection, such as bronchitis or pneumonia.', 'One possible observation about persistent cough could be that it is accompanied by other symptoms such as shortness of breath, chest pain, or fever, which could indicate a more serious underlying condition such as pneumonia or bronchitis.', 'One possible observation about a persistent cough could be that it is accompanied by other symptoms such as wheezing, shortness of breath, or chest pain, indicating a possible respiratory condition such as bronchitis or asthma.']
Analysis: Summary of observations: One possible observation about a persistent cough is that it could be a sign of a respiratory infection, such as bronchitis or pneumonia. | One possible observation about persistent cough could be that it is accompanied by other symptoms such as shortness of breath, chest pain, or fever, which could indicate a more serious underlying 

In [ ]:
# Scenario: AI-Assisted Email Workflow (Question-Based)
# Context
# A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
# The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
# and sent or rejected.

# 1. State Definition
# The assistant maintains a notebook-like state:
# - task → The subject or purpose of the email (e.g., "Q3 Report").
# - draft → The AI-generated email draft.
# - approved → A flag indicating whether the human reviewer has approved the draft.

# 2. Workflow (Graph of States)
# Each email task flows through nodes:
# - Draft Node
# - AI generates a draft email based on the task.
# - Updates draft.
# - Review Node (Interrupt)
# - Execution pauses here.
# - Human reviewer inspects the draft and decides whether to approve or reject.
# - Updates approved.
# - Send Node
# - If approved = True → Email is sent.
# - If approved = False → Email is rejected.
# - Updates task with final status (SENT or REJECTED).
# - End Node
# - Workflow completes.

# 3. Example Flow
# - Employee: "Draft an email for the Q3 Report."
# - State: task = "Q3 Report"
# - Assistant drafts:
# Dear User,
# Regarding: Q3 Report
# [AI drafted content]
# - Human reviews → Approves draft (approved = True)
# - Assistant sends → task = "SENT: Q3 Report"
# - Final Output: ✅ Email delivered.

# 👉 Scenario Question:
# "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
# then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
#  and human oversight in the process?"
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# 🔐 API CONFIG (OpenRouter)
API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get("OPENAI_API_KEY")

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# 1️⃣ State Definition
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2️⃣ Nodes

# Draft Node (AI generates email)
def draft_email(state: EmailState):
    print(f"📝 Drafting email for task: {state['task']}")

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json={
            "model": "openai/gpt-3.5-turbo",
            "messages": [
                {
                    "role": "user",
                    "content": f"Write a professional email about: {state['task']}"
                }
            ],
            "temperature": 0.7
        }
    )

    data = response.json()
    draft = data["choices"][0]["message"]["content"]

    return {"draft": draft}


# Review Node (Interrupt point)
def human_review(state: EmailState):
    print("\n📧 Draft ready for review:\n")
    print(state["draft"])
    print("\n⏸ Waiting for human approval...\n")
    return {}


# Send Node
def send_email(state: EmailState):
    if state.get("approved", False):
        print("✅ Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("❌ Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}


# 3️⃣ Build Graph
g = StateGraph(EmailState)

g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

# Flow
g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

# 4️⃣ Memory (needed for pause/resume)
checkpointer = MemorySaver()

app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)

#  Run Workflow

thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Generate draft (PAUSES before review)
app.invoke(
    {"task": "Q3 Report", "draft": "", "approved": False},
    thread
)

# Step 2: Human approval (resume execution)
# Change True → False to test rejection
app.invoke(
    {"approved": True},
    thread
)

📝 Drafting email for task: Q3 Report
📝 Drafting email for task: Q3 Report


{'task': 'Q3 Report',
 'draft': 'Subject: Quarterly Report for Q3\n\nDear Team,\n\nI hope this email finds you well. I am writing to provide you with the Quarterly Report for Q3. The report contains detailed information on our performance and achievements during the third quarter of the year.\n\nIn the report, you will find a summary of our financial results, key milestones reached, and an analysis of our progress towards our strategic goals. It also includes a breakdown of our sales figures, market trends, and a review of our marketing and operational activities.\n\nI encourage you to review the report carefully and reach out to me if you have any questions or require further clarification on any of the information presented. Your feedback and insights are invaluable in helping us make informed decisions and drive our business forward.\n\nThank you for your hard work and dedication during this quarter. I am proud of what we have accomplished together and look forward to continuing our

In [ ]:
# Scenario Question "Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review, and then
# either publishes or rejects them. How would you structure the state and workflow graph to ensure accountability and human oversight in
# the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

#  API CONFIG (OpenRouter)
API_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = userdata.get("OPENAI_API_KEY")

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

#  State Definition
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool
    status: str

#  Nodes

# Draft Node (AI generates report)
def draft_report(state: ReportState):
    print(f" Generating report for: {state['task']}")

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json={
            "model": "openai/gpt-3.5-turbo",
            "messages": [
                {
                    "role": "user",
                    "content": f"Create a structured professional report on: {state['task']}"
                }
            ],
            "temperature": 0.7
        }
    )

    data = response.json()
    draft = data["choices"][0]["message"]["content"]

    return {"draft": draft}


# Review Node (Interrupt)
def review_node(state: ReportState):
    print("\n Draft ready for review:\n")
    print(state["draft"])
    print("\n⏸ Waiting for human approval...\n")
    return {}


# Decision Function (Routing)
def approval_check(state: ReportState):
    if state.get("approved", False):
        return "publish"
    else:
        return "reject"


# Publish Node
def publish_report(state: ReportState):
    print(" Report Approved and Published.")
    return {"status": f"PUBLISHED: {state['task']}"}


# Reject Node
def reject_report(state: ReportState):
    print(" Report Rejected.")
    return {"status": f"REJECTED: {state['task']}"}


#  Build Graph
g = StateGraph(ReportState)

g.add_node("draft", draft_report)
g.add_node("review", review_node)
g.add_node("publish", publish_report)
g.add_node("reject", reject_report)

# Flow
g.set_entry_point("draft")
g.add_edge("draft", "review")

# Conditional routing after review
g.add_conditional_edges(
    "review",
    approval_check,
    {
        "publish": "publish",
        "reject": "reject"
    }
)

g.add_edge("publish", END)
g.add_edge("reject", END)

#  Memory (for pause/resume)
checkpointer = MemorySaver()

app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)

#  Run Workflow

thread = {"configurable": {"thread_id": "report-1"}}

#  Step 1: Generate Draft (pauses before review)
app.invoke(
    {"task": "Monthly Sales Report", "draft": "", "approved": False, "status": ""},
    thread
)

#  Step 2: Human decision (True = publish, False = reject)
app.invoke(
    {"approved": True},   # change to False to test rejection
    thread
)

 Generating report for: Monthly Sales Report
 Generating report for: Monthly Sales Report


{'task': 'Monthly Sales Report',
 'draft': 'Title: Monthly Sales Report\n\nDate: [Month, Year]\n\nPrepared by: [Your Name]\n\n1. Executive Summary:\nThis report provides an overview of the sales performance for the month of [Month, Year]. It includes a summary of total sales, sales by product category, sales by region, and comparison to previous months and targets.\n\n2. Total Sales Performance:\n- Total Sales for the month: $[Total Sales Amount]\n- Percentage increase/decrease compared to previous month: [Percentage Increase/Decrease]\n- Percentage increase/decrease compared to target: [Percentage Increase/Decrease]\n\n3. Sales by Product Category:\n- Category 1: $[Sales Amount]\n- Category 2: $[Sales Amount]\n- Category 3: $[Sales Amount]\n- Category 4: $[Sales Amount]\n- Category 5: $[Sales Amount]\n\n4. Sales by Region:\n- Region 1: $[Sales Amount]\n- Region 2: $[Sales Amount]\n- Region 3: $[Sales Amount]\n- Region 4: $[Sales Amount]\n- Region 5: $[Sales Amount]\n\n5. Comparison to